# Analisis Exploratorio de Datos (EDA) y Pipeline de Enriquecimiento (NLP + Metricas Relativas)

## Contexto y Objetivo del EDA
Los eventos gestionados en **TuBoleta** cuentan con una gran heterogeneidad en los nombres comerciales de sus localidades (*VIP, General, Platino, Palcos, Platea, Occidental, Oriental, etc.*), las cuales cambian segun el recinto (*venue*) y el promotor.

En este cuaderno realizamos el **Analisis Exploratorio de Datos (EDA)** conectando directamente los **Modulos 1 (NLP)** y **2 (Ingenieria de Caracteristicas Relativas)** de nuestro codigo fuente en `src/`:
1. **Modulo 1 - Descomposicion NLP (`src.nlp_utils`)**: Supresion de ruido publicitario/marketing, extraccion de 17 variables estructurales/espaciales y normalizacion de texto.
2. **Modulo 2 - Metricas Relativas por Evento (`src.feature_engineering`)**: Desacoplar la escala del venue calculando `percentil_precio_evento`, `ratio_precio_max`, `peso_aforo` y `tasa_ocupacion`.
3. **Analisis Exploratorio y Visualizaciones**: Evaluar la separabilidad natural de los datos antes de pasar a la clusterizacion (Modulo 3).

## 1. Configuracion del Entorno e Importacion de Librerias

In [ ]:
import os
import sys
sys.path.append("../")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importacion de funciones modulares de los Modulos 1 y 2
from src.nlp_utils import (
    limpiar_ruido_marketing,
    extraer_atributos_estructurales,
    pipeline_procesamiento_nlp
)
from src.feature_engineering import (
    filtrar_consistencia_localidades,
    calcular_metricas_relativas,
    preparar_dataset_enriquecido
)

# Configuracion de estilos visuales
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
sns.set_palette("viridis")

print("Librerias y modulos de src/ cargados exitosamente.")

## 2. Carga del Conjunto de Datos Raw
Cargamos directamente el dataset descargado de Azure en `data/raw/localidades_eda.parquet`.

In [ ]:
data_path = "../data/raw/localidades_eda.parquet" if os.path.exists("../data/raw/localidades_eda.parquet") else "data/raw/localidades_eda.parquet"
df_raw = pd.read_parquet(data_path)

print(f"Dimensiones del dataset original: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
df_raw.head(4)

## 3. Calidad de Datos y Filtrado de Consistencia (Modulo 2 - Paso 1)
Utilizamos `filtrar_consistencia_localidades()` para garantizar la validez fisica del aforo y montos no negativos.

In [ ]:
print("=== ESTADISTICAS NUMERICAS ORIGINALES ===")
key_numeric = ["performance_quota", "dn_quota", "med_unit_amt_itx", "ave_unit_amt_itx", "net_sold_p_qty", "net_sold_c_qty"]
display(df_raw[key_numeric].describe().T)

# Aplicar filtro de consistencia modular
df_clean = filtrar_consistencia_localidades(df_raw)

print(f"\nFilas iniciales: {len(df_raw):,}")
print(f"Filas tras filtro de consistencia: {len(df_clean):,} (Conservado el {len(df_clean)/len(df_raw)*100:.2f}% con calidad certificada)")

## 4. Analisis y Descomposicion NLP (Modulo 1 - `src.nlp_utils`)

Evaluamos la limpieza de ruido publicitario (`limpiar_ruido_marketing`) y la extraccion de 17 variables binarias estructurales (`extraer_atributos_estructurales`).

In [ ]:
# Prueba visual de limpieza de ruido publicitario sobre casos reales de TuBoleta
ejemplos_marketing = [
    "PALCOS CANTINERO - LLEGO EL PODER",
    "PALCOS EL REENCUENTRO - BOMBASTIK",
    "SIGO INVICTO - SILLAS VIP",
    "PISO 3 - 302 - 306 & 314 - 318",
    "PALCO NEGRA PULOY SENDE",
    "EXPERIENCIA PASEO DE LA AURORA PLATINO",
    "ORIENTAL ALTA FAMILIAR - LIBRE DE ALCOHOL"
]

print("=== PRUEBA DEL PIPELINE DE LIMPIEZA NLP ===")
for ej in ejemplos_marketing:
    limpio = limpiar_ruido_marketing(ej)
    print(f"Original: {ej:<45} -> Limpio: {limpio}")

In [ ]:
# Ejecutar el pipeline de NLP sobre todo el dataset limpio
df_nlp = pipeline_procesamiento_nlp(df_clean, col_nombre="logical_seat_category")

# Grafico 1: Frecuencia de tags estructurales detectados
tag_cols = [c for c in df_nlp.columns if c.startswith("tag_")]
tag_counts = df_nlp[tag_cols].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=tag_counts.values, y=tag_counts.index, palette="mako")
plt.title("Grafico 1: Frecuencia de Atributos Estructurales, Espaciales y Restricciones Extraidos (NLP)")
plt.xlabel("Total de Ocurrencias en el Catalogo")
plt.ylabel("Tag Estructural (NLP)")
plt.tight_layout()
plt.show()

### Analisis del Grafico 1:
De las 33,878 localidades evaluadas, la etiqueta mas predominante es `tag_general` con **15,386 apariciones (45.4%)**, seguida de niveles de teatro y recintos cerrados como `tag_balcon` (**4,319**), `tag_platea` (**4,037**) y `tag_palco` (**3,581**). En el ambito espacial, las orientaciones con mayor frecuencia son `tag_occidental` (1,004) y `tag_norte` (810). Las restricciones de acceso (`tag_familiar`, `tag_movilidad_reducida`, `tag_menores`) representan nichos menores a 100 registros pero de alto valor operacional.

#### Conclusiones del Grafico 1:
1. **Predominio de Localidades Masivas:** Casi la mitad del inventario historico de TuBoleta corresponde a localidades de acceso general o admision unica, lo que exige que el modelo sea capaz de diferenciar una 'General de Estadio' frente a una 'Entrada General de Teatro'.
2. **Alta Especializacion en Teatros y Arenas:** La gran presencia de tokens como `balcon`, `platea` y `palco` (mas de 11,900 registros combinados) confirma que la mayoria de eventos con silleteria numerada tienen estructuras verticales escalonadas.
3. **Efectividad del Pipeline Regex/NER:** La deteccion de 17 etiquetas permitio convertir texto desestructurado en variables numericas computables, cubriendo los ejes de jerarquia de nivel, orientacion en el recinto y condiciones de acceso.

## 5. Ingenieria de Caracteristicas Relativas por Evento (Modulo 2 - `src.feature_engineering`)

Calculamos las variables normalizadas por funcion (`t_performance_id`):
- **`peso_aforo`**: dn_quota / performance_quota (% de capacidad total que ocupa la localidad).
- **`ratio_precio_max`**: med_unit_amt_itx / max(med_unit_amt_itx) del evento (precio relativo vs la localidad mas cara).
- **`percentil_precio_evento`**: Posicion ordinal de precio en la funcion (de 0 a 1).
- **`tasa_ocupacion`**: (net_sold_p_qty + net_sold_c_qty) / dn_quota (absorcion historica de demanda).

In [ ]:
# Aplicar el calculo de metricas relativas modulares
df_enriquecido = calcular_metricas_relativas(df_nlp)

print(f"Dataset enriquecido: {df_enriquecido.shape[0]:,} filas y {df_enriquecido.shape[1]} columnas.")
df_enriquecido[["product", "logical_seat_category", "texto_limpio", "med_unit_amt_itx", "ratio_precio_max", "percentil_precio_evento", "peso_aforo", "tasa_ocupacion"]].head(6)

## 6. Analisis de Distribuciones de las Variables del Espacio Mixto

In [ ]:
# Grafico 2: Panel 2x2 de distribuciones de variables relativas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Ratio de Precio Maximo
sns.histplot(df_enriquecido["ratio_precio_max"], bins=30, kde=True, ax=axes[0, 0], color="royalblue")
axes[0, 0].set_title("Distribucion del Ratio de Precio (vs Maximo del Evento)")
axes[0, 0].set_xlabel("Ratio de Precio Relativo (1.0 = Mas cara del evento)")

# 2. Peso de Aforo
sns.histplot(df_enriquecido["peso_aforo"], bins=30, kde=True, ax=axes[0, 1], color="crimson")
axes[0, 1].set_title("Distribucion del Peso de Aforo (% de Capacidad Total)")
axes[0, 1].set_xlabel("Peso de Aforo (dn_quota / performance_quota)")

# 3. Tasa de Ocupacion
sns.histplot(df_enriquecido["tasa_ocupacion"], bins=30, kde=True, ax=axes[1, 0], color="seagreen")
axes[1, 0].set_title("Distribucion de la Tasa de Ocupacion Historica")
axes[1, 0].set_xlabel("Tasa de Ocupacion (0 = 0%, 1 = 100% Sold Out)")

# 4. Percentil de Precio por Evento
sns.histplot(df_enriquecido["percentil_precio_evento"], bins=30, kde=True, ax=axes[1, 1], color="darkorange")
axes[1, 1].set_title("Distribucion del Percentil de Precio por Evento")
axes[1, 1].set_xlabel("Percentil Relativo de Precio (0 a 1)")

plt.suptitle("Grafico 2: Distribuciones de Variables Estructurales y de Rendimiento", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Analisis del Grafico 2:
El `ratio_precio_max` muestra una concentracion alta en 1.0 (mediana = 1.0, media = 0.803), lo que indica que en eventos de pocas localidades o tarifa unica, el ratio se satura en el tope. El `peso_aforo` presenta una distribucion bimodal: una concentracion de zonas exclusivas de aforo reducido (percentil 25 = 0.096, menos del 10% del venue), y eventos de admision unica donde una sola localidad ocupa el 100% del aforo. La `tasa_ocupacion` media se situa en 18.1% (mediana = 10.8%), con un percentil 90 en 47.7%.

#### Conclusiones del Grafico 2:
1. **Desacople Exitoso de la Moneda:** El uso de ratios (0.0 a 1.0) resolvio la distorsion del precio en pesos COP, permitiendo comparar espectaculos de $50,000 COP con eventos internacionales de $1,500,000 COP bajo la misma regla de exclusividad.
2. **Identificacion Inmediata de Zonas Selectas:** El 25% de las localidades del catalogo (`peso_aforo` < 0.096) corresponden a zonas de capacidad muy reducida, candidatas naturales a palcos o VIP.
3. **Comportamiento Comercial Asimetrico:** La tasa de ocupacion permite incorporar la velocidad de absorcion comercial como variable de diferenciacion de demanda entre zonas populares y zonas de alta rotacion.

## 7. Comparacion Bivariada por Arquetipos Semanticos Extraidos

In [ ]:
# Comparacion cuantitativa de variables relativas segun tags NLP
tag_summary = []
for tag in ["tag_palco", "tag_vip", "tag_platea", "tag_preferencial", "tag_general", "tag_balcon", "tag_vista_parcial"]:
    subset = df_enriquecido[df_enriquecido[tag] == 1]
    tag_summary.append({
        "Etiqueta NLP": tag.replace("tag_", "").upper(),
        "Total Localidades": len(subset),
        "Ratio Precio Promedio": round(subset["ratio_precio_max"].mean(), 2),
        "Mediana Precio COP": int(subset["med_unit_amt_itx"].median()),
        "Peso Aforo Medio (%)": round(subset["peso_aforo"].mean() * 100, 1),
        "Tasa Ocupacion Media (%)": round(subset["tasa_ocupacion"].mean() * 100, 1)
    })

df_tag_summary = pd.DataFrame(tag_summary).set_index("Etiqueta NLP")
display(df_tag_summary)

In [ ]:
# Grafico 3: Boxplots de Precio Relativo y Peso de Aforo por etiqueta NLP
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

tag_data = []
for tag in ["tag_palco", "tag_vip", "tag_preferencial", "tag_platea", "tag_general", "tag_balcon"]:
    temp = df_enriquecido[df_enriquecido[tag] == 1].copy()
    temp["tag_name"] = tag.replace("tag_", "").upper()
    tag_data.append(temp)
df_tags_long = pd.concat(tag_data, ignore_index=True)

sns.boxplot(data=df_tags_long, x="tag_name", y="ratio_precio_max", palette="Set2", ax=axes[0])
axes[0].set_title("Ratio de Precio Relativo por Tag Estructural (NLP)")
axes[0].set_xlabel("Atributo Estructural")
axes[0].set_ylabel("Ratio de Precio (vs Maximo)")

sns.boxplot(data=df_tags_long, x="tag_name", y="peso_aforo", palette="Set2", ax=axes[1])
axes[1].set_title("Peso de Aforo (% de Capacidad Total) por Tag Estructural (NLP)")
axes[1].set_xlabel("Atributo Estructural")
axes[1].set_ylabel("Peso de Aforo (%)")

plt.suptitle("Grafico 3: Comparacion Bivariada de Precio y Aforo por Atributo Estructural", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Analisis del Grafico 3:
`PALCO` y `VIP` presentan las medianas de `peso_aforo` mas bajas del catalogo (4.0% y 4.4% del recinto), mientras mantienen ratios de precio promedio de 0.685 y 0.676 con bigotes superiores en 1.0. `PREFERENCIAL` y `PLATEA` tienen precios relativos altos (medianas de 0.923 y 0.875) y aforos medios (15.7% a 17.4% del recinto). `BALCON` presenta el precio relativo mas bajo entre zonas de teatro (media de 0.503, mediana de 0.481) y `GENERAL` absorbe la mayor porcion de capacidad (mediana de aforo = 100%).

#### Conclusiones del Grafico 3:
1. **Confirmacion de Jerarquia Fisica:** La semantica extraida por el NLP se correlaciona con la fisica del recinto: los palcos ocupan fracciones minimas de aforo mientras que las generales absorben el volumen.
2. **Validacion de la Platea como Zona Preferente:** Las plateas y preferenciales se ubican en el rango superior de precios de cada espectaculo, validando su rol como el escalon intermedio-alto de demanda.
3. **El Balcon como Opcion Popular en Recintos Cerrados:** Los balcones y pisos altos registran sistematicamente un ratio de precio 50% inferior al de la platea del mismo teatro, confirmando su rol de accesibilidad economica.

## 8. Matriz de Correlaciones Numericas y Ratios

In [ ]:
# Grafico 4: Matriz de correlaciones
corr_cols = [
    "med_unit_amt_itx", "ratio_precio_max", "ratio_precio_mean", "percentil_precio_evento",
    "dn_quota", "peso_aforo", "tasa_ocupacion", "tasa_venta_paga", "ratio_cortesias"
]

plt.figure(figsize=(10, 8))
sns.heatmap(df_enriquecido[corr_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", cbar=True, vmin=-1, vmax=1)
plt.title("Grafico 4: Matriz de Correlacion de Variables Estructurales y de Rendimiento")
plt.tight_layout()
plt.show()

### Analisis del Grafico 4:
La correlacion entre el precio nominal en pesos (`med_unit_amt_itx`) y el `ratio_precio_max` es practicamente nula (r = -0.034), demostrando que el precio nominal en COP no describe la posicion de exclusividad de la boleta. El `peso_aforo` se correlaciona negativamente con el `ratio_precio_max` (r = -0.248) y con el `ratio_cortesias` (r = -0.250). La `tasa_ocupacion` tiene una alta correlacion positiva con `tasa_venta_paga` (r = 0.82) y moderada con `ratio_cortesias` (r = 0.354).

#### Conclusiones del Grafico 4:
1. **Independencia del Precio Nominal:** Al tener correlacion cercana a cero con los ratios relativos, se ratifica que usar el precio en COP aisladamente distorsiona los clusters y que la normalizacion por evento era indispensable.
2. **Ley de Oferta y Demanda en el Aforo:** A mayor peso de aforo de una localidad dentro del evento, menor tiende a ser su ratio de precio relativo, lo que respalda la separacion entre zonas masivas y exclusivas.
3. **No Redundancia en el Espacio Mixto:** Ninguna pareja de variables estructurales presenta colinealidad perfecta (|r| < 0.85), garantizando que cada dimension aporta informacion independiente al modelo.

## 9. Mapa de Separabilidad Espacial (Espacio de Clustering)

In [ ]:
# Grafico 5: Dispersion 2D de cuadrantes de demanda
plt.figure(figsize=(12, 7))
scatter = plt.scatter(
    df_enriquecido["peso_aforo"],
    df_enriquecido["ratio_precio_max"],
    c=df_enriquecido["tasa_ocupacion"],
    cmap="plasma",
    alpha=0.4,
    edgecolors="none",
    s=25
)
plt.colorbar(scatter, label="Tasa de Ocupacion (0 = Venta baja, 1 = Sold Out)")
plt.title("Grafico 5: Espacio de Clustering - Ratio de Precio Relativo vs Peso de Aforo")
plt.xlabel("Peso de Aforo (% de Capacidad Total del Evento)")
plt.ylabel("Ratio de Precio Relativo (1.0 = Localidad mas cara del evento)")

# Anotaciones de los 4 cuadrantes teoricos de demanda
plt.axvline(x=0.25, color="gray", linestyle="--", alpha=0.5)
plt.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)

plt.text(0.02, 0.95, "VIP / Palcos / Premium\n(Alto Precio + Bajo Aforo)", fontsize=11, weight="bold", bbox=dict(facecolor="white", alpha=0.7))
plt.text(0.02, 0.25, "Popular Exclusiva / Lateral\n(Bajo Precio + Bajo Aforo)", fontsize=11, weight="bold", bbox=dict(facecolor="white", alpha=0.7))
plt.text(0.55, 0.90, "Preferencial Masiva\n(Alto Precio + Gran Aforo)", fontsize=11, weight="bold", bbox=dict(facecolor="white", alpha=0.7))
plt.text(0.55, 0.25, "Grada General / Masiva\n(Bajo Precio + Gran Aforo)", fontsize=11, weight="bold", bbox=dict(facecolor="white", alpha=0.7))

plt.tight_layout()
plt.show()

### Analisis del Grafico 5:
El grafico proyecta la separacion natural en 4 cuadrantes: el cuadrante superior izquierdo (Bajo Aforo < 25%, Alto Precio > 0.5) poblado por zonas VIP, palcos y plateas preferenciales con ocupaciones altas; el cuadrante inferior izquierdo (Bajo Aforo < 25%, Bajo Precio < 0.5) ocupado por balcones altos, laterales y vistas restringidas; el cuadrante superior derecho (Alto Aforo > 25%, Alto Precio > 0.5) con preferenciales masivas; y el cuadrante inferior derecho (Alto Aforo > 25%, Bajo Precio < 0.5) con gradas generales y populares.

#### Conclusiones del Grafico 5:
1. **Separabilidad Geometrica Evidente:** Los datos no forman una masa amorfa, sino cuatro concentraciones espaciales que validan el uso de algoritmos basados en distancias (K-Means).
2. **Diferenciacion por Demanda (Ocupacion):** La escala de color evidencia que las zonas del cuadrante superior izquierdo presentan las mayores tasas de agotamiento de boleteria.
3. **Soporte Visual para Decisiones de Negocio:** Este mapa permite a los equipos de pricing y producto ubicar cualquier nueva localidad de un promotor y entender de inmediato a que cuadrante pertenece.

## 10. Conclusiones Generales del EDA y Pase al Modelo de Clustering (Modulo 3)

### Hallazgos Principales del EDA:
1. **Efectividad del Modulo 1 (NLP)**: La eliminacion de ruido publicitario (*patrocinios, nombres de gira como 'LLEGO EL PODER'*) y la extraccion de tags estructurales permite capturar la jerarquia real de las localidades sin crear falsos duplicados.
2. **Poder de Normalizacion del Modulo 2 (Metricas Relativas)**: `percentil_precio_evento` y `ratio_precio_max` unifican eventos de estadio de 46,000 personas con teatros de 300 personas en una escala comparable (0.0 a 1.0).
3. **Separabilidad de Cuadrantes**: Las dimensiones `(peso_aforo, ratio_precio_max, tasa_ocupacion)` delimitan con claridad los 4 arquetipos de demanda fundamentales.

Siguiente Paso: Abrir el cuaderno **`notebooks/02_clustering_espacio_mixto.ipynb`** para construir el espacio vectorial mixto de 37 dimensiones y entrenar el modelo de Machine Learning (Modulo 3).